# 🌊 AquaYOLO: Underwater Object Detection & Benchmark Training

This notebook runs training, evaluation, and complete metric visualization for the **AquaYOLO** marine object detection project in Google Colab with GPU acceleration.

### 📌 Features in this Notebook:
- **Google Drive Integration**: Seamlessly connect your dataset stored in Google Drive.
- **High-Accuracy YOLO Training**: Transfer learning with marine and COCO pretrained backbones, mosaic augmentation, and cosine learning rate schedule.
- **Metric Visualizations**: Displays all YOLO-generated metric graphs directly in notebook outputs:
  - `results.png` (Train/Val loss, Precision, Recall, mAP@0.50, mAP@0.50:0.95)
  - `confusion_matrix.png` and `confusion_matrix_normalized.png`
  - `PR_curve.png` (Precision-Recall curve)
  - `F1_curve.png` (F1 Score vs Confidence)
  - `P_curve.png` and `R_curve.png`
  - `val_batch0_pred.jpg` (Visual prediction comparisons)
- **AquaYOLO Custom Attention**: Run and evaluate the custom AquaYOLO model featuring SAC (Spectral Attenuation Compensator) and TCG (Turbidity Contrast Gate).

## 🚀 Step 1: Check GPU Runtime
Make sure you have selected **GPU** as your runtime type (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 📂 Step 2: Mount Google Drive
Mount your Google Drive to load the dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 3: Clone AquaYOLO Repository & Install Dependencies

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/Aravind8833/AquaYOLO.git
%cd /content/AquaYOLO

# Install dependencies
!pip install -q ultralytics opencv-python matplotlib seaborn pyyaml pandas pillow
print('\nEnvironment ready!')

## 📁 Step 4: Configure Dataset from Google Drive
Specify the path to your dataset in Google Drive. You can provide either:
1. A **Folder path** (e.g. `/content/drive/MyDrive/AquaYOLO_data`), OR
2. A **Zip file path** (e.g. `/content/drive/MyDrive/AquaYOLO_data.zip`), which unzips to local Colab storage for ultra-fast GPU training.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import yaml

# === UPDATE THIS PATH TO YOUR DATASET IN GOOGLE DRIVE ===
DRIVE_DATASET_SOURCE = "/content/drive/MyDrive/AquaYOLO_26class_balanced"  # or .zip file

LOCAL_DATASET_DIR = Path("/content/dataset")
LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

source_path = Path(DRIVE_DATASET_SOURCE)
if source_path.suffix == ".zip":
    print(f"Extracting {source_path} to {LOCAL_DATASET_DIR}...")
    with zipfile.ZipFile(source_path, "r") as zip_ref:
        zip_ref.extractall(LOCAL_DATASET_DIR)
    print("Extraction complete!")
elif source_path.is_dir():
    print(f"Using dataset directory from Google Drive: {source_path}")
    LOCAL_DATASET_DIR = source_path
else:
    print(f"Path not found yet: {source_path}. Please check your Drive path.")

# Locate or create data.yaml
yaml_candidates = list(LOCAL_DATASET_DIR.rglob("data.yaml"))
if yaml_candidates:
    DATA_YAML_PATH = yaml_candidates[0]
    print(f"Found dataset config at: {DATA_YAML_PATH}")
else:
    # Create data.yaml pointing to images
    DATA_YAML_PATH = LOCAL_DATASET_DIR / "data.yaml"
    classes = [
        "jellyfish", "penguin", "puffin", "starfish", "stingray",
        "basking_shark", "blacktip_reef_shark", "great_white_shark",
        "hammerhead_shark", "nurse_shark", "oceanic_whitetip_shark",
        "thresher_shark", "whale_shark", "whitetip_reef_shark",
        "ballan_wrasse", "cuckoo_wrasse", "corkwing_wrasse",
        "goldsinny_wrasse", "rock_cook", "atlantic_cod",
        "pollack", "whiting", "spiny_dogfish", "brown_crab",
        "two_spot_goby", "human"
    ]
    data_dict = {
        "path": str(LOCAL_DATASET_DIR.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: name for i, name in enumerate(classes)}
    }
    with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_dict, f, sort_keys=False)
    print(f"Created fresh dataset config at: {DATA_YAML_PATH}")

## 🏋️ Step 5: High-Accuracy Training (YOLO Benchmark)
Trains YOLO with transfer learning, cosine learning rate decay, and mosaic augmentation to achieve high accuracy (>75% mAP50).

In [ ]:
# Run high-accuracy training
!python train_yolo.py \
    --data "{DATA_YAML_PATH}" \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 16 \
    --imgsz 640 \
    --project /content/AquaYOLO/runs/yolo_train \
    --name colab_run

## 📊 Step 6: Display All Generated Metric Graphs
This displays every single metric graph generated by YOLO directly in the notebook output:
1. **Training & Validation Loss / Metrics Curve** (`results.png`)
2. **Confusion Matrix & Normalized Confusion Matrix** (`confusion_matrix.png`)
3. **Precision-Recall Curve** (`PR_curve.png`)
4. **F1-Confidence Curve** (`F1_curve.png`)
5. **Precision & Recall Curves** (`P_curve.png`, `R_curve.png`)
6. **Validation Ground Truth vs Model Predictions** (`val_batch0_labels.jpg`, `val_batch0_pred.jpg`)

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

RUN_DIR = Path("/content/AquaYOLO/runs/yolo_train/colab_run")

def show_graph(image_filename, title):
    p = RUN_DIR / image_filename
    if p.exists():
        print("=" * 75)
        print(f"{title}: {p.name}")
        print("=" * 75)
        display(Image(filename=str(p), width=900))
    else:
        print(f"{image_filename} not generated yet.")

# 1. Training & Validation Results (Losses, Precision, Recall, mAP50, mAP50-95)
show_graph("results.png", "TRAINING & VALIDATION METRICS OVER EPOCHS")

# 2. Precision-Recall Curve
show_graph("PR_curve.png", "PRECISION-RECALL (PR) CURVE")

# 3. F1-Confidence Curve
show_graph("F1_curve.png", "F1-SCORE VS CONFIDENCE CURVE")

# 4. Precision & Recall vs Confidence Curves
show_graph("P_curve.png", "PRECISION VS CONFIDENCE CURVE")
show_graph("R_curve.png", "RECALL VS CONFIDENCE CURVE")

# 5. Confusion Matrices
show_graph("confusion_matrix.png", "CONFUSION MATRIX")
show_graph("confusion_matrix_normalized.png", "NORMALIZED CONFUSION MATRIX")

# 6. Visual Detections on Validation Set (Ground Truth vs Model Prediction)
show_graph("val_batch0_labels.jpg", "GROUND TRUTH LABELS")
show_graph("val_batch0_pred.jpg", "MODEL PREDICTIONS ON VALIDATION BATCH")

## 📋 Step 7: View Numerical Metrics Table (Epoch by Epoch)

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print("\n--- FINAL EPOCHS SUMMARY ---")
    display(df.tail(10))
else:
    print("results.csv not found.")

## 🧠 Step 8: (Optional) Train Custom AquaYOLO Architecture
AquaYOLO model with CSPDarknet backbone, Spectral Attenuation Compensator (SAC), and Turbidity Contrast Gate (TCG).

In [ ]:
# Train custom AquaYOLO model
!python src/aquayolo/train.py \
    --data "{DATA_YAML_PATH.parent}" \
    --epochs 30 \
    --batch 4 \
    --size 640 \
    --lr 0.0001 \
    --out ./runs/aquayolo_custom_colab

# Show AquaYOLO loss curves
aqua_loss_plot = Path("./runs/aquayolo_custom_colab/loss_curve.png")
if aqua_loss_plot.exists():
    display(Image(filename=str(aqua_loss_plot), width=800))
aqua_comp_plot = Path("./runs/aquayolo_custom_colab/loss_components.png")
if aqua_comp_plot.exists():
    display(Image(filename=str(aqua_comp_plot), width=900))